# iTransformer forecasting — out-of-fold sentiment

Three properties the protocol guarantees:

1. **Sentiment is out-of-fold.** Every headline is scored by a FinBERT model trained only on
   strictly earlier data (`sentiment_analysis/oof_finbert_scoring.ipynb`,
   `sentiment_analysis/oof_score_full_stream.ipynb`), rather than by a single checkpoint applied to
   its own training headlines.
2. **Model selection never sees a test regime.** Hyperparameters and epoch counts are chosen on a
   120-day inner-validation slice ending at the training cut-off.
3. **The sample starts when sentiment exists.** Out-of-fold scoring produces nothing before
   2020-01-01, and zero-filling those hours would conflate "no news", "neutral news" and "no model
   available". All four feature sets start together at 2020-01-01.

It also reports naive benchmarks, three seeds with mean ± sd, and Diebold–Mariano tests against
**both** a zero-change forecast and the `Price + volume` baseline, separately per regime.

## Protocol

The protocol is *imported* from `lstm_models/lstm_common.py` through `itransformer_common.py` —
feature construction, lookback, split dates, scaling, metrics, benchmarks and tests are the same
objects the LSTM study uses, so "same target, lookback, periods and preprocessing" is guaranteed by
construction rather than asserted. Only the forecaster differs.

| | |
|---|---|
| Feature sets | `price+volume` (4), `+sentiment` (5), `+technicals` (10), `full` (11) |
| Lookback / target | 168 h; next-hour change of the BTCUSDT close, in USD |
| Sample | 2020-01-01 → end of common coverage, one window for all four sets |
| Split | training ends 2024-04-30; May 2024 is a buffer month |
| Test regimes | uptrend 2024-06 → 2025-01; sideways 2025-01 → 2025-06 |
| Scaling | inputs and target standardised on training-period statistics |
| Selection | Optuna (50 trials) then epoch count, both on the inner-validation slice |
| Seeds | 3 per feature set; tables report mean ± sd |
| Benchmarks | zero-change, training historical mean, random walk, AR(1), AR(24) |
| Tests | Diebold–Mariano on absolute errors, Newey–West variance, HLN correction, BH across the family |

In [ ]:
import json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import itransformer_common as ic
from itransformer_common import (
    FEATURE_SET_ORDER, NICE_NAME, TRAIN_END, INNER_VAL_START, UPTREND, SIDEWAYS,
    ANALYSIS_START, SENT_FILE_TRAIN, OUT, CKPT_DIR,
    build_feature_sets, restrict_to_common, common_window, make_windows, regime_mask,
    metrics, ar_forecast, dm_test, bh_qvalues, train_eval, tune,
    ITransformerPredictor, DEVICE,
)

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)

SEEDS = [0, 1, 2]
RUN_TUNING = True          # False -> reuse out/itransformer_hparams.json
MAX_EPOCHS = 60
PATIENCE = 8

print("device:", DEVICE)
print("sentiment file:", SENT_FILE_TRAIN)
print("analysis starts:", ANALYSIS_START)

In [ ]:
HOURLY = build_feature_sets(SENT_FILE_TRAIN, verbose=True)
HOURLY_C = restrict_to_common(HOURLY)
lo, hi = common_window(HOURLY)
sideways_end = min(SIDEWAYS[1], hi)

print(f"\ncommon window: {lo} -> {hi}")
if sideways_end < SIDEWAYS[1]:
    print(f"NOTE: coverage ends {hi.date()}; sideways truncated to "
          f"{SIDEWAYS[0].date()}..{sideways_end.date()} for every feature set")

rows = []
for n, h in HOURLY_C.items():
    rows.append(dict(feature_set=n, variates=h.shape[1], rows=len(h),
                     fit=int((h.index < INNER_VAL_START).sum()),
                     inner_val=int(((h.index >= INNER_VAL_START) & (h.index < TRAIN_END)).sum()),
                     uptrend=int(regime_mask(h.index, "Uptrend").sum()),
                     sideways=int((regime_mask(h.index, "Sideways") & (h.index < sideways_end)).sum())))
split = pd.DataFrame(rows)
split.to_csv(OUT / "itransformer_data_split.csv", index=False)
print(); print(split.to_string(index=False))

## Hyperparameter search

50 Optuna trials per feature set, 5 epochs each on a reduced corpus, scored by inner-validation MAE
with median pruning — the same budget the published study used, but on a slice that ends before the
test regimes begin.

In [ ]:
HP_FILE = OUT / "itransformer_hparams.json"

if RUN_TUNING:
    t0 = time.time()
    HP = {n: tune(HOURLY_C[n], n) for n in FEATURE_SET_ORDER}
    HP_FILE.write_text(json.dumps(HP, indent=2))
    print(f"\ntuning took {(time.time()-t0)/60:.1f} min -> {HP_FILE}")
else:
    HP = json.loads(HP_FILE.read_text())
    print("reusing", HP_FILE)

print()
print(pd.DataFrame(HP).T.to_string())

## Training

Per feature set and seed: choose the epoch count on the inner-validation slice, then refit on the
full training period and predict both regimes. The refit models are saved as a seed ensemble for the
backtesting notebook.

In [ ]:
fit_mask   = lambda idx: idx < INNER_VAL_START
inner_mask = lambda idx: (idx >= INNER_VAL_START) & (idx < TRAIN_END)
train_mask = lambda idx: idx < TRAIN_END
test_mask  = lambda idx: (idx >= UPTREND[0]) & (idx < sideways_end)

preds, epoch_rows, histories = {}, [], {}
t0 = time.time()

for name in FEATURE_SET_ORDER:
    dfh, hp, states = HOURLY_C[name], HP[name], []
    for seed in SEEDS:
        sel = train_eval(dfh, fit_mask, inner_mask, hp, epochs=MAX_EPOCHS,
                         seed=seed, track=True, patience=PATIENCE)
        n_ep = max(1, sel["best_epoch"] + 1)
        histories[(name, seed)] = sel["history"]

        run = train_eval(dfh, train_mask, test_mask, hp, epochs=n_ep,
                         seed=seed, return_model=True)
        preds[(name, seed)] = run
        states.append(run["state_dict"])
        epoch_rows.append(dict(feature_set=name, seed=seed, epochs=n_ep,
                               inner_val_mae=sel["history"][sel["best_epoch"]]))
        print(f"{name:26s} seed={seed}  epochs={n_ep:2d}  inner-val MAE={sel['history'][sel['best_epoch']]:8.2f}")

    ITransformerPredictor(states, preds[(name, SEEDS[0])]["scaler"], hp,
                          preds[(name, SEEDS[0])]["columns"], dfh.shape[1],
                          list(dfh.columns).index("price_change_1h")
                          ).save(CKPT_DIR / f"itransformer_{ic.FOLDER_NAME[name]}.pt")

epoch_sel = pd.DataFrame(epoch_rows)
epoch_sel.to_csv(OUT / "itransformer_epoch_selection.csv", index=False)
print(f"\ntotal training time: {(time.time()-t0)/60:.1f} min")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 3.6), sharey=True)
for ax, name in zip(axes, FEATURE_SET_ORDER):
    for seed in SEEDS:
        h = histories[(name, seed)]
        ax.plot(range(1, len(h) + 1), h, lw=1.2, label=f"seed {seed}")
        ax.axvline(np.argmin(h) + 1, color="grey", ls=":", lw=0.8)
    ax.set_title(NICE_NAME[name], fontsize=10)
    ax.set_xlabel("epoch")
axes[0].set_ylabel("inner-validation MAE (USD)")
axes[0].legend(fontsize=8, frameon=False)
fig.suptitle("iTransformer epoch selection on the inner-validation slice", fontsize=11)
fig.tight_layout()
fig.savefig(OUT / "epochs_itransformer.png", dpi=150, bbox_inches="tight")
print("wrote", OUT / "epochs_itransformer.png")

## Forecasting performance

In [ ]:
perf_rows = []
for name in FEATURE_SET_ORDER:
    for regime in ("Uptrend", "Sideways"):
        per_seed = []
        for seed in SEEDS:
            r = preds[(name, seed)]
            m = regime_mask(r["idx"], regime)
            if regime == "Sideways":
                m &= r["idx"] < sideways_end
            per_seed.append(metrics(r["pred"][m], r["y"][m], r["close_prev"][m], name, regime))
        keys = ("MAE_usd", "RMSE_usd", "MAE_pct", "corr", "dir_acc", "mean_pred", "std_pred")
        agg = {k: float(np.mean([d[k] for d in per_seed])) for k in keys}
        agg |= {f"{k}_sd": float(np.std([d[k] for d in per_seed]))
                for k in ("MAE_usd", "RMSE_usd", "corr", "dir_acc")}
        perf_rows.append(dict(regime=regime, model=name, n=per_seed[0]["n"],
                              seeds=len(SEEDS), **agg))

perf = pd.DataFrame(perf_rows)
perf.to_csv(OUT / "itransformer_forecasting_performance.csv", index=False)
print(perf[["regime", "model", "n", "MAE_usd", "MAE_usd_sd", "RMSE_usd", "RMSE_usd_sd",
            "corr", "corr_sd", "dir_acc", "dir_acc_sd"]].to_string(
      index=False, float_format=lambda v: f"{v:.4f}"))

## Naive benchmarks

Zero-change, the training-period historical mean, a random walk on the target (next change = last
change), and AR(1)/AR(24) fitted by OLS on the training window only. All are evaluated on the same
index as the models.

In [ ]:
ref = HOURLY_C["full"]
_, y_ref, idx_ref, cp_ref, _ = make_windows(ref)
base_rows = []

for regime in ("Uptrend", "Sideways"):
    rm = lambda idx, rg=regime: (regime_mask(idx, rg) & (idx < sideways_end)
                                 if rg == "Sideways" else regime_mask(idx, rg))
    m = rm(idx_ref)
    y, cp = y_ref[m], cp_ref[m]
    hist_mean = float(y_ref[idx_ref < TRAIN_END].mean())

    base_rows.append(metrics(np.zeros_like(y), y, cp, "zero change", regime))
    base_rows.append(metrics(np.full_like(y, hist_mean), y, cp, "historical mean (train)", regime))
    prev = np.concatenate([[np.nan], y[:-1]]); ok = np.isfinite(prev)
    base_rows.append(metrics(prev[ok], y[ok], cp[ok], "random walk (last change)", regime))
    for p in (1, 24):
        pr, ya, ev = ar_forecast(y_ref, idx_ref, lambda idx: idx < TRAIN_END, rm, p)
        base_rows.append(metrics(pr, ya, cp_ref[ev], f"AR({p})", regime))

baselines = pd.DataFrame(base_rows)
baselines.to_csv(OUT / "itransformer_baselines.csv", index=False)
print(baselines[["regime", "model", "n", "MAE_usd", "RMSE_usd", "corr", "dir_acc"]].to_string(
      index=False, float_format=lambda v: f"{v:.4f}"))

zc = baselines[baselines.model == "zero change"].set_index("regime")["MAE_usd"]
print("\nMAE relative to a zero-change forecast (1.00 = matched it):")
for regime in ("Uptrend", "Sideways"):
    sub = perf[perf.regime == regime]
    line = "  ".join(f"{NICE_NAME[r.model]}: {r.MAE_usd / zc[regime]:.3f}x" for r in sub.itertuples())
    print(f"  {regime:9s} {line}")

## Diebold–Mariano tests

On absolute errors of the **seed-averaged** forecast, with Newey–West variance and the
Harvey–Leybourne–Newbold small-sample correction, Benjamini–Hochberg across the family.

Two comparison families, as requested: every model against a zero-change forecast, and every model
against the `Price + volume` baseline. Negative DM means the first model has the lower absolute
error.

In [ ]:
frames = {}
for name in FEATURE_SET_ORDER:
    P = pd.DataFrame({s: pd.Series(preds[(name, s)]["pred"], index=preds[(name, s)]["idx"])
                      for s in SEEDS})
    y = pd.Series(preds[(name, SEEDS[0])]["y"], index=preds[(name, SEEDS[0])]["idx"])
    frames[name] = pd.DataFrame({"err": P.mean(axis=1) - y, "y": y})

common = None
for f in frames.values():
    common = f.index if common is None else common.intersection(f.index)
print(f"common evaluation index across all four feature sets: {len(common):,} hours")

err = pd.DataFrame({n: f.loc[common, "err"] for n, f in frames.items()})
err["zero change"] = -frames["full"].loc[common, "y"]

dm_rows = []
for regime in ("Uptrend", "Sideways"):
    m = regime_mask(err.index, regime)
    if regime == "Sideways":
        m &= err.index < sideways_end
    sub = err[m]
    for name in FEATURE_SET_ORDER:
        r = dm_test(sub[name].to_numpy(), sub["zero change"].to_numpy(), loss="abs")
        dm_rows.append(dict(regime=regime, family="vs zero change",
                            comparison=f"{NICE_NAME[name]} vs zero change",
                            d_mean=r["dbar"], DM=r["dm"], p=r["p"], n=r["n"]))
    for name in FEATURE_SET_ORDER[1:]:
        r = dm_test(sub[name].to_numpy(), sub["price+volume"].to_numpy(), loss="abs")
        dm_rows.append(dict(regime=regime, family="vs price+volume",
                            comparison=f"{NICE_NAME[name]} vs {NICE_NAME['price+volume']}",
                            d_mean=r["dbar"], DM=r["dm"], p=r["p"], n=r["n"]))

dm = pd.DataFrame(dm_rows)
dm["q_BH"] = bh_qvalues(dm["p"])
dm.to_csv(OUT / "itransformer_dm_tests.csv", index=False)
print()
print(dm.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print("\nNegative DM => the first model has the LOWER absolute error.")

## LaTeX drop-ins

In [ ]:
def _tex(df, cols, header, caption, label, fmt):
    body = "\n".join(
        "        " + " & ".join(fmt(r, c) for c in cols) + r" \\" for _, r in df.iterrows())
    return (f"\\begin{{table}}[htbp]\n\\centering\n\\caption{{{caption}}}\n"
            f"\\label{{{label}}}\n\\begin{{tabular}}{{{'l' * 2 + 'r' * (len(cols) - 2)}}}\n"
            f"\\hline\n{header} \\\\\n\\hline\n{body}\n\\hline\n"
            f"\\end{{tabular}}\n\\end{{table}}\n")

pf = perf.copy()
pf["model"] = pf["model"].map(NICE_NAME)
t1 = _tex(pf, ["regime", "model", "n", "MAE_usd", "RMSE_usd", "corr", "dir_acc"],
          "Regime & Feature set & $n$ & MAE (USD) & RMSE (USD) & Corr. & Dir. acc. (\\%)",
          "iTransformer forecasting performance with out-of-fold sentiment. "
          "Mean $\\pm$ sd over three seeds.",
          "tab:itransformer_forecasting_oof",
          lambda r, c: (f"{r[c]:.1f} $\\pm$ {r[c + '_sd']:.1f}" if c in ("MAE_usd", "RMSE_usd")
                        else f"{r[c]:.4f} $\\pm$ {r['corr_sd']:.4f}" if c == "corr"
                        else f"{r[c]:.2f}" if c == "dir_acc"
                        else f"{r[c]:,}" if c == "n" else str(r[c])))
(OUT / "table_itransformer_forecasting.tex").write_text(t1)

t2 = _tex(dm, ["regime", "comparison", "n", "DM", "p", "q_BH"],
          "Regime & Comparison & $n$ & DM & $p$ & $q_{BH}$",
          "Diebold--Mariano tests on absolute errors of the seed-averaged iTransformer forecast.",
          "tab:itransformer_dm_oof",
          lambda r, c: (f"{r[c]:,}" if c == "n" else f"{r[c]:.3f}" if c == "DM"
                        else f"{r[c]:.4f}" if c in ("p", "q_BH") else str(r[c])))
(OUT / "table_itransformer_dm.tex").write_text(t2)

bl = baselines.copy()
t3 = _tex(bl, ["regime", "model", "n", "MAE_usd", "RMSE_usd", "corr", "dir_acc"],
          "Regime & Benchmark & $n$ & MAE (USD) & RMSE (USD) & Corr. & Dir. acc. (\\%)",
          "Naive forecasting benchmarks on the evaluation windows.",
          "tab:itransformer_baselines",
          lambda r, c: (f"{r[c]:,}" if c == "n" else f"{r[c]:.1f}" if c in ("MAE_usd", "RMSE_usd")
                        else f"{r[c]:.4f}" if c == "corr" else f"{r[c]:.2f}" if c == "dir_acc"
                        else str(r[c])))
(OUT / "table_itransformer_baselines.tex").write_text(t3)

with pd.ExcelWriter(OUT / "itransformer_forecasting_tables.xlsx") as xl:
    perf.to_excel(xl, sheet_name="forecasting", index=False)
    baselines.to_excel(xl, sheet_name="baselines", index=False)
    dm.to_excel(xl, sheet_name="dm_tests", index=False)
    epoch_sel.to_excel(xl, sheet_name="epoch_selection", index=False)
    split.to_excel(xl, sheet_name="data_split", index=False)

# idx as ISO strings: an int64 view of a tz-aware index loses its unit,
# which silently breaks the downstream merge in results/.
np.savez_compressed(OUT / "itransformer_predictions.npz",
                    **{f"{n}__{k}": v for n in FEATURE_SET_ORDER
                       for k, v in [(f"pred_seed{s}", preds[(n, s)]["pred"]) for s in SEEDS]
                       + [("y", preds[(n, SEEDS[0])]["y"]),
                          ("idx", preds[(n, SEEDS[0])]["idx"].astype(str))]})

for f in ["table_itransformer_forecasting.tex", "table_itransformer_dm.tex",
          "table_itransformer_baselines.tex", "itransformer_forecasting_tables.xlsx",
          "itransformer_predictions.npz"]:
    print("wrote", OUT / f)